# Day 05 - AI Clothing Store Chatbot

This notebook builds a friendly chatbot with **OpenAI** and **Gradio**.

## Learning goals
- Load credentials securely with `python-dotenv`
- Send chat history and system instructions to an LLM
- Stream responses into a polished browser interface
- Add business rules that change the assistant's behavior

> **Security:** Never commit or print the full API key.

## 1. Import libraries

- `os` reads environment variables.
- `load_dotenv` loads the local `.env` file.
- `OpenAI` calls the chat-completions API.
- `gradio` creates the interactive web UI.

In [1]:
import os

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

c:\AI_ENGINEER_CORE\Agentic AI - Core Track\Day_05\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the API key

Create a `.env` file next to this notebook:

```env
OPENAI_API_KEY=your-api-key-here
```

The code prints only the first eight characters for debugging.

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY is not set. Add it to a .env file or environment variables."
    )

print(f"OpenAI API Key is set (starts with: {openai_api_key[:8]}...)")

OpenAI API Key is set (starts with: sk-proj-...)


## 3. Configure the model

`OpenAI()` uses the key loaded above. `MODEL` controls which model answers requests.

Replace `gpt-6-luna` with a model available to your OpenAI account if needed.

In [3]:
openai = OpenAI(api_key=openai_api_key)
MODEL = "gpt-6-luna"

system_message = """
You are a helpful assistant in a clothes store.
Try to gently encourage the customer to try items that are on sale.
Hats are 60% off, and most other items are 50% off.
Be friendly, concise, and honest.
"""

## 4. Normalize chat history

Gradio 6 sends history as message dictionaries. This helper also supports older tuple-style history, where each pair contains a user message and an assistant message.

The normalized result can be passed directly to OpenAI.

In [4]:
def normalize_history(history):
    """Convert Gradio history into OpenAI message dictionaries."""
    normalized = []

    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in {"user", "assistant", "system"} and content:
                normalized.append({"role": role, "content": content})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_message, assistant_message = item
            if user_message:
                normalized.append({"role": "user", "content": str(user_message)})
            if assistant_message:
                normalized.append({"role": "assistant", "content": str(assistant_message)})

    return normalized

## 5. Create the streaming chat function

The function builds the complete message list in this order:

1. System instructions
2. Previous conversation history
3. The latest user message

It then streams the response and yields the accumulated text so Gradio updates the chat bubble continuously.

In [5]:
def chat(message, history):
    relevant_system_message = system_message

    if "shoe" in message.lower():
        relevant_system_message += "\nShoes are not on sale today. Remind the customer to look at hats."

    if "belt" in message.lower():
        relevant_system_message += "\nThe store does not sell belts. Clearly say this and suggest another sale item."

    messages = (
        [{"role": "system", "content": relevant_system_message}]
        + normalize_history(history)
        + [{"role": "user", "content": message}]
    )

    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        reasoning_effort="none",
        stream=True,
    )

    response = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            response += delta
            yield response

## 6. Launch the polished chatbot

This fixes the reported error: Gradio 6.32.0 does **not** accept `type="messages"` in `gr.ChatInterface`.

The title, description, placeholder, examples, and soft theme make the interface easier to use and more visually polished.

In [6]:
demo = gr.ChatInterface(
    fn=chat,
    title="Clothing Store AI Assistant",
    description="Get friendly recommendations and learn about today's sales.",
    textbox=gr.Textbox(placeholder="How can I help you shop?", lines=2),
    examples=[
        "Do you have shoes?",
        "Do you sell belts?",
        "What hat should I buy?",
        "What is on sale?",
    ],
)

demo.launch(theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
